# Air Quality Trend Estimator — Cleaned Model Pipeline

This notebook prepares the India city-day dataset for the project's chosen inputs and outputs.

**Input features:** `PM2.5`, `PM10`, `NO`, `NO2`, `SO2`  
**Targets:** numeric `AQI` (regression) and `AQI_Bucket` (classification)  
**Kept as metadata:** `Date`  
**Removed:** `City` and all other columns.

Rows without an AQI value are removed. Missing values in the five input features are retained at preprocessing time and imputed using medians learned from the training data only. This preserves more labeled records and avoids test-data leakage.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)
from pathlib import Path

# Update this path if your CSV is stored elsewhere.
CSV_PATH = "city_day.csv"

df = pd.read_csv(CSV_PATH)
print("Original dataset shape:", df.shape)
print(df.columns.tolist())

In [ ]:
# Keep only the fields needed by this project.
feature_cols = ["PM2.5", "PM10", "NO", "NO2", "SO2"]
required_cols = ["Date"] + feature_cols + ["AQI", "AQI_Bucket"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

model_df = df[required_cols].copy()
model_df["Date"] = pd.to_datetime(model_df["Date"], errors="coerce")
model_df["AQI"] = pd.to_numeric(model_df["AQI"], errors="coerce")

# Explicit user requirement: discard rows where AQI is absent.
before = len(model_df)
model_df = model_df.dropna(subset=["AQI", "Date"]).copy()
model_df["AQI_Bucket"] = model_df["AQI_Bucket"].astype("string").str.strip()
model_df = model_df.dropna(subset=["AQI_Bucket"])
model_df = model_df[model_df["AQI_Bucket"] != ""]
model_df = model_df.sort_values("Date").reset_index(drop=True)

print(f"Rows before filtering: {before:,}")
print(f"Rows after removing missing AQI/date/bucket: {len(model_df):,}")
print("Columns retained:", model_df.columns.tolist())
print("\nMissing values remaining:")
print(model_df.isna().sum())
model_df.head()

## Train/test split by date

The date stays in the cleaned data for plotting and chronological evaluation, but it is **not** one of the five model inputs. The test set contains the latest 20% of unique dates, so evaluation is closer to forecasting future records than a random split.

In [ ]:
unique_dates = np.sort(model_df["Date"].unique())
if len(unique_dates) < 5:
    raise ValueError("Not enough unique dates for a chronological train/test split.")

cutoff_idx = int(len(unique_dates) * 0.8)
cutoff_idx = min(max(cutoff_idx, 1), len(unique_dates) - 1)
test_start_date = pd.Timestamp(unique_dates[cutoff_idx])

train_df = model_df[model_df["Date"] < test_start_date].copy()
test_df = model_df[model_df["Date"] >= test_start_date].copy()

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train_aqi = train_df["AQI"]
y_test_aqi = test_df["AQI"]
y_train_bucket = train_df["AQI_Bucket"].astype(str)
y_test_bucket = test_df["AQI_Bucket"].astype(str)

print("Test period starts:", test_start_date.date())
print("Training rows:", len(train_df), "| Test rows:", len(test_df))
print("Training date range:", train_df["Date"].min().date(), "to", train_df["Date"].max().date())
print("Test date range:", test_df["Date"].min().date(), "to", test_df["Date"].max().date())

## Model 1 — Predict numeric AQI

Median imputation is fitted on the training subset only. Random Forest is used as the initial model to stay close to the original notebook; a baseline Linear Regression can be compared later.

In [ ]:
aqi_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

aqi_model.fit(X_train, y_train_aqi)
aqi_predictions = aqi_model.predict(X_test)

mae = mean_absolute_error(y_test_aqi, aqi_predictions)
rmse = np.sqrt(mean_squared_error(y_test_aqi, aqi_predictions))
r2 = r2_score(y_test_aqi, aqi_predictions)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

## Model 2 — Predict AQI bucket

AQI bucket is a categorical target, so it needs a classification model rather than a regression model. This trains a separate Random Forest classifier using the same five pollutant inputs.

In [ ]:
bucket_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2,
        class_weight="balanced"
    ))
])

bucket_model.fit(X_train, y_train_bucket)
bucket_predictions = bucket_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test_bucket, bucket_predictions):.3f}")
print("\nClassification report:")
print(classification_report(y_test_bucket, bucket_predictions, zero_division=0))

## Inspect actual versus predicted values

In [ ]:
comparison = test_df[["Date", "AQI", "AQI_Bucket"]].copy()
comparison["Predicted_AQI"] = aqi_predictions
comparison["Predicted_AQI_Bucket"] = bucket_predictions
comparison = comparison.rename(columns={
    "AQI": "Actual_AQI",
    "AQI_Bucket": "Actual_AQI_Bucket"
})
comparison.head(20)

## Save the trained models

These files can be loaded by the Python API. Keep the feature order consistent with `feature_cols`.

In [ ]:
OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(aqi_model, OUTPUT_DIR / "aqi_regressor_5_features.pkl")
joblib.dump(bucket_model, OUTPUT_DIR / "aqi_bucket_classifier_5_features.pkl")
joblib.dump(feature_cols, OUTPUT_DIR / "aqi_feature_columns.pkl")

print("Saved:")
print(OUTPUT_DIR / "aqi_regressor_5_features.pkl")
print(OUTPUT_DIR / "aqi_bucket_classifier_5_features.pkl")
print(OUTPUT_DIR / "aqi_feature_columns.pkl")

## Example prediction

Supply values for all five input features. Missing input values are allowed because the saved pipeline applies its fitted median imputer. The example values below are placeholders; replace them with realistic pollutant measurements in the dataset's units.

In [ ]:
sample = pd.DataFrame([{
    "PM2.5": 26,
    "PM10": 50,
    "NO": 1,
    "NO2": 19,
    "SO2": 3
}])[feature_cols]

predicted_aqi = float(aqi_model.predict(sample)[0])
predicted_bucket = str(bucket_model.predict(sample)[0])

print(f"Predicted AQI: {predicted_aqi:.2f}")
print(f"Predicted AQI bucket: {predicted_bucket}")